In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from pathlib import Path
import nilearn
import gc
from nilearn.glm.first_level import FirstLevelModel
from nilearn.image import load_img, concat_imgs

def get_rp24(confound_file):
    confounds = pd.read_csv(confound_file, sep="\t")

    base_cols   = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]
    deriv_cols  = [f"{c}_derivative1" for c in base_cols]
    power2_cols = [f"{c}_power2" for c in base_cols]
    deriv2_cols = [f"{c}_derivative1_power2" for c in base_cols]
    rp24_cols   = base_cols + deriv_cols + power2_cols + deriv2_cols

    missing = [c for c in rp24_cols if c not in confounds.columns]
    if missing:
        raise ValueError(f"missing rp-24 columns: {missing}")

    rp_24 = confounds[rp24_cols].copy()
    rp_24 = rp_24.fillna(0.0)  # first derivative row often has NaNs
    assert rp_24.shape[1] == 24, f"expected 24 columns in rp_24, got {rp_24.shape[1]}"
    return rp_24

def build_trials_table(timing_file, behavior_file):
    """
    Build a trial table for nilearn GLM.

    Rules:
    - Keep only Narrative and Decision trials
    - Use timing file onsets
    - Narrative duration = timing-file duration
    - Decision duration = reaction_time from behavior file
    - If no response or missing RT, duration = 0
    """
    # -------------------- load timing --------------------
    timing = pd.read_excel(timing_file)
    timing = timing.sort_values("onset").reset_index(drop=True)

    # keep only modeled events
    timing = timing[timing["trial_type"].isin(["Narrative", "Decision"])].copy()
    timing = timing.reset_index(drop=True)

    # force numeric where needed
    timing["trial_num"] = pd.to_numeric(timing["trial_num"], errors="coerce")
    timing["onset"] = pd.to_numeric(timing["onset"], errors="coerce")
    timing["duration"] = pd.to_numeric(timing["duration"], errors="coerce")

    if "decision_num" not in timing.columns:
        timing["decision_num"] = np.nan
    else:
        timing["decision_num"] = pd.to_numeric(timing["decision_num"], errors="coerce")

    # default modeled duration = original timing duration
    timing["duration_model"] = timing["duration"]

    # -------------------- load behavior --------------------
    beh = pd.read_excel(behavior_file)
    beh = beh.sort_values("decision_num").reset_index(drop=True)

    beh["decision_num"] = pd.to_numeric(beh["decision_num"], errors="coerce")
    beh["reaction_time"] = pd.to_numeric(beh["reaction_time"], errors="coerce")

    # missing RT -> 0
    beh["reaction_time"] = beh["reaction_time"].fillna(0.0)

    if "responded" in beh.columns:
        if beh["responded"].dtype == object:
            beh["responded"] = beh["responded"].astype(str).str.upper().eq("TRUE")
        else:
            beh["responded"] = beh["responded"].astype(bool)
    else:
        beh["responded"] = True

    # if not responded, modeled duration = 0
    beh["duration_model"] = beh["reaction_time"]
    beh.loc[~beh["responded"], "duration_model"] = 0.0

    # also make absolutely sure duration_model has no NaNs
    beh["duration_model"] = beh["duration_model"].fillna(0.0)

    # -------------------- map RTs onto decision trials --------------------
    rt_map = beh.set_index("decision_num")["duration_model"]

    is_dec = timing["trial_type"].eq("Decision")
    timing.loc[is_dec, "duration_model"] = timing.loc[is_dec, "decision_num"].map(rt_map)

    # any unmatched or missing decision durations -> 0
    timing.loc[is_dec, "duration_model"] = timing.loc[is_dec, "duration_model"].fillna(0.0)

    # final duration column used for modeling
    timing["duration"] = timing["duration_model"]
    timing = timing.drop(columns=["duration_model"])

    # final checks
    assert timing["onset"].notna().all(), "Non-finite onsets found"
    assert timing["duration"].notna().all(), "Non-finite durations found"

    return timing

def make_lss_events(trials, target_decision_num):
    """
    Build the 3-regressor LSS events table:
      - target_decision: the one target decision trial
      - other_decision: all other decision trials
      - other_narrative: all narrative trials
    """
    events = trials[["onset", "duration", "trial_type", "decision_num"]].copy()
    events = events[events["trial_type"].isin(["Narrative", "Decision"])].copy()

    events.loc[events["trial_type"] == "Narrative", "trial_type"] = "other_narrative"
    events.loc[events["trial_type"] == "Decision",  "trial_type"] = "other_decision"

    is_target = events["decision_num"].eq(target_decision_num)
    if is_target.sum() != 1:
        raise ValueError(
            f"Expected exactly 1 target decision for decision_num={target_decision_num}, "
            f"found {is_target.sum()}"
        )

    events.loc[is_target, "trial_type"] = "target_decision"
    return events[["onset", "duration", "trial_type"]].copy()

def force_nrows(df, n_rows):
    """Truncate or zero-pad confounds to match n_scans."""
    df = df.copy().reset_index(drop=True)
    if len(df) == n_rows:
        return df
    if len(df) > n_rows:
        return df.iloc[:n_rows].reset_index(drop=True)

    pad = pd.DataFrame(0.0, index=np.arange(n_rows - len(df)), columns=df.columns)
    return pd.concat([df, pad], axis=0, ignore_index=True)

def safe_delete(path):
    path = Path(path)
    try:
        if path.exists():
            path.unlink()
    except Exception:
        pass

def file_is_good(path, min_bytes=1024, check_load=True):
    """
    Basic file check:
    - exists
    - non-trivial size
    - can be opened as a NIfTI (optional)
    """
    path = Path(path)
    if not path.exists() or not path.is_file():
        return False

    try:
        if path.stat().st_size < min_bytes:
            return False
    except Exception:
        return False

    if check_load:
        try:
            _ = load_img(str(path))
        except Exception:
            return False

    return True

def find_first_existing(paths, require_good=False):
    for p in paths:
        if p is None:
            continue
        p = Path(p)
        if p.exists():
            if require_good:
                if file_is_good(p):
                    return p
            else:
                return p
    return None

def any_existing(paths):
    return any(Path(p).exists() for p in paths if p is not None)

def delete_all(paths):
    for p in paths:
        if p is not None:
            safe_delete(p)

def find_task_specific_mask_file(sub_id, func_dir, anat_dir):
    """
    Prefer the task-specific functional-space brain mask.
    Fall back to broader patterns only if needed.
    """
    candidates = [
        next(func_dir.glob(f"{sub_id}_task-socialnav_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii*"), None),
        next(func_dir.glob(f"{sub_id}_task-socialnav*_desc-brain_mask.nii*"), None),
        next(func_dir.glob(f"{sub_id}*_task-socialnav*_desc-brain_mask.nii*"), None),
        next(anat_dir.glob(f"{sub_id}_desc-brain_mask.nii*"), None),
        next(anat_dir.glob(f"{sub_id}*_desc-brain_mask.nii*"), None),
    ]
    return find_first_existing(candidates, require_good=False)

def get_trial_output_paths(trialmaps_dir, trial_idx):
    beta_file = trialmaps_dir / f"decision_{trial_idx:04d}_beta.nii.gz"
    t_file    = trialmaps_dir / f"decision_{trial_idx:04d}_t.nii.gz"
    return beta_file, t_file

def merge_trialmaps_to_4d(img_files, out_file):
    if len(img_files) == 0:
        raise ValueError(f"No images provided for merge: {out_file}")

    missing = [str(p) for p in img_files if not file_is_good(p)]
    if missing:
        raise FileNotFoundError("Missing or invalid trial maps before merge:\n" + "\n".join(missing))

    merged = concat_imgs([str(p) for p in img_files], auto_resample=False)
    merged.to_filename(str(out_file))

    merged_shape = load_img(str(out_file)).shape
    n_vols = 1 if len(merged_shape) == 3 else merged_shape[3]
    if n_vols != len(img_files):
        raise RuntimeError(
            f"Merged file has wrong number of volumes: expected {len(img_files)}, found {n_vols}"
        )

#------------------------------- main runner

def run_lss_subject(
    sub_id,
    preprc_dir,
    behav_dir,
    timing_file,
    glm_dir,
    tr=1.0,
    slice_time_ref=0.5,
    high_pass=1/128,
):
    """
    Runs decision-trial LSS for one subject.

    Restart logic:
    - If BOTH merged outputs already exist (.nii.gz or .nii) -> skip subject.
    - If merged outputs do not exist -> reuse any valid cached per-trial maps.
    - If a trial has only one of beta/t -> delete both and rerun that trial.
    - Per-trial maps are kept after success so restarts can resume or re-merge.
    """

    subj_dir = Path(preprc_dir) / sub_id
    func_dir = subj_dir / "func"
    anat_dir = subj_dir / "anat"

    out_dir = Path(glm_dir) / sub_id
    out_dir.mkdir(parents=True, exist_ok=True)

    tmp_root = out_dir / "_tmp_lss_decision"
    trialmaps_dir = tmp_root / "trialmaps_3d"
    trialmaps_dir.mkdir(parents=True, exist_ok=True)

    # Final merged outputs: accept either .nii.gz or .nii as "done"
    merged_beta_gz = out_dir / "decision_trials_beta.nii.gz"
    merged_beta_nii = out_dir / "decision_trials_beta.nii"
    merged_t_gz = out_dir / "decision_trials_t.nii.gz"
    merged_t_nii = out_dir / "decision_trials_t.nii"

    merged_beta_variants = [merged_beta_gz, merged_beta_nii]
    merged_t_variants = [merged_t_gz, merged_t_nii]

    beta_final_good = find_first_existing(merged_beta_variants, require_good=True)
    t_final_good = find_first_existing(merged_t_variants, require_good=True)

    # -------------------- done? --------------------
    if beta_final_good is not None and t_final_good is not None:
        print(f"[{sub_id}] merged outputs already exist -> skipping")
        print(f"  beta: {beta_final_good}")
        print(f"  t:    {t_final_good}")
        return

    # If one merged output exists but the other does not, or if only invalid partials exist,
    # delete all final-output variants and rebuild from per-trial maps.
    beta_any_exists = any_existing(merged_beta_variants)
    t_any_exists = any_existing(merged_t_variants)
    if beta_any_exists or t_any_exists:
        print(f"[{sub_id}] partial or invalid merged outputs found -> deleting final-output variants and rebuilding")
        delete_all(merged_beta_variants + merged_t_variants)

    # -------------------- find files --------------------
    behav_file = next(Path(behav_dir).glob(f"{sub_id}.xlsx"), None)
    func_file = next(func_dir.glob(f"{sub_id}_task-socialnav*_desc-preproc_bold.nii*"), None)
    anat_file = next(anat_dir.glob(f"{sub_id}*_desc-preproc_T1w.nii*"), None)
    mask_file = find_task_specific_mask_file(sub_id, func_dir, anat_dir)
    confound_file = next(func_dir.glob(f"{sub_id}_task-socialnav*_desc-confounds_timeseries.tsv"), None)

    if behav_file is None:
        raise FileNotFoundError(f"{sub_id}: could not find behavioral file in {behav_dir}")
    if func_file is None:
        raise FileNotFoundError(f"{sub_id}: could not find func file in {func_dir}")
    if anat_file is None:
        raise FileNotFoundError(f"{sub_id}: could not find anat file in {anat_dir}")
    if confound_file is None:
        raise FileNotFoundError(f"{sub_id}: could not find confounds file in {func_dir}")
    if mask_file is None:
        raise FileNotFoundError(
            f"{sub_id}: could not find task-specific brain mask in {func_dir} "
            f"(or fallback mask in {anat_dir})"
        )

    print(f"\n[{sub_id}]")
    print(f"  behav:     {behav_file}")
    print(f"  func:      {func_file}")
    print(f"  anat:      {anat_file}")
    print(f"  mask:      {mask_file}")
    print(f"  confounds: {confound_file}")

    # -------------------- confounds --------------------
    rp_24 = get_rp24(confound_file)
    n_scans = load_img(str(func_file)).shape[-1]

    if len(rp_24) != n_scans:
        print(f"[{sub_id}] confounds rows ({len(rp_24)}) != n_scans ({n_scans}) -> trunc/pad")
        rp_24 = force_nrows(rp_24, n_scans)

    # -------------------- trials --------------------
    trials = build_trials_table(timing_file, behav_file)
    decision_trials = trials.loc[trials["trial_type"] == "Decision", ["trial_num", "decision_num", "onset", "duration"]].copy()
    decision_trials = decision_trials.reset_index(drop=True)

    if len(decision_trials) == 0:
        raise RuntimeError(f"{sub_id}: no decision trials found")

    # Save the exact order used for per-trial output naming
    decision_index_file = out_dir / "decision_trial_index.tsv"
    decision_trials_for_save = decision_trials.copy()
    decision_trials_for_save.insert(0, "lss_trial_idx", np.arange(1, len(decision_trials_for_save) + 1))
    decision_trials_for_save.to_csv(decision_index_file, sep="\t", index=False)

    expected_beta_files = []
    expected_t_files = []

    # -------------------- loop over decision trials --------------------
    # BUG FIX: iterate over actual decision_num values, not row index + 1
    for ii, target_decision_num in enumerate(decision_trials["decision_num"].tolist(), start=1):
        beta_file, t_file = get_trial_output_paths(trialmaps_dir, ii)
        expected_beta_files.append(beta_file)
        expected_t_files.append(t_file)

        # Reuse complete existing per-trial maps
        if file_is_good(beta_file) and file_is_good(t_file):
            print(f"[{sub_id}] reuse trial {ii:03d}/{len(decision_trials)} (decision_num={target_decision_num})")
            continue

        # If partial or bad trial output exists, delete both and rerun
        if beta_file.exists():
            safe_delete(beta_file)
        if t_file.exists():
            safe_delete(t_file)

        print(f"[{sub_id}] fit   trial {ii:03d}/{len(decision_trials)} (decision_num={target_decision_num})")

        events_lss = make_lss_events(trials, target_decision_num)

        glm = FirstLevelModel(
            t_r=tr,
            slice_time_ref=slice_time_ref,  # middle of TR; approx SPM middle-slice microtime reference
            hrf_model="spm",
            drift_model="cosine",
            high_pass=high_pass,
            mask_img=str(mask_file),        # task-specific mask
            smoothing_fwhm=None,            # unsmoothed
            signal_scaling=False,
            minimize_memory=False,
        )

        try:
            glm = glm.fit(
                run_imgs=str(func_file),
                events=events_lss,
                confounds=rp_24,
            )

            design = glm.design_matrices_[0]
            if "target_decision" not in design.columns:
                raise RuntimeError(
                    f"'target_decision' missing from design columns:\n{design.columns.tolist()}"
                )

            # target-trial beta and t-stat maps
            beta_img = glm.compute_contrast("target_decision", output_type="effect_size")
            t_img = glm.compute_contrast("target_decision", output_type="stat")

            # save one image per decision trial
            beta_img.to_filename(str(beta_file))
            t_img.to_filename(str(t_file))

            if not file_is_good(beta_file) or not file_is_good(t_file):
                raise RuntimeError("Saved per-trial output failed file check")

        except Exception:
            # Clean partial outputs for this trial, then re-raise
            safe_delete(beta_file)
            safe_delete(t_file)
            raise

        finally:
            for varname in ["glm", "design", "beta_img", "t_img", "events_lss"]:
                if varname in locals():
                    del locals()[varname]
            gc.collect()

    # -------------------- verify all per-trial maps exist --------------------
    missing_beta = [str(p) for p in expected_beta_files if not file_is_good(p)]
    missing_t = [str(p) for p in expected_t_files if not file_is_good(p)]

    if missing_beta or missing_t:
        msg = []
        if missing_beta:
            msg.append("Missing beta maps:\n" + "\n".join(missing_beta))
        if missing_t:
            msg.append("Missing t maps:\n" + "\n".join(missing_t))
        raise RuntimeError("\n\n".join(msg))

    # -------------------- merge to final 4D outputs --------------------
    # Write gzip outputs by default
    safe_delete(merged_beta_gz)
    safe_delete(merged_beta_nii)
    safe_delete(merged_t_gz)
    safe_delete(merged_t_nii)

    print(f"[{sub_id}] merging beta maps -> {merged_beta_gz.name}")
    merge_trialmaps_to_4d(expected_beta_files, merged_beta_gz)

    print(f"[{sub_id}] merging t maps -> {merged_t_gz.name}")
    merge_trialmaps_to_4d(expected_t_files, merged_t_gz)

    print(f"[{sub_id}] done")
    print(f"  merged beta: {merged_beta_gz}")
    print(f"  merged t:    {merged_t_gz}")


main_dir = '/Users/matty_gee/Desktop/Social/SocialCUD'

# preprocessed directories
behav_dir  = f'{main_dir}/data/preprocessed/behavior'
preprc_dir = f'{main_dir}/data/preprocessed/fmriprep/derivatives-fmap/fmriprep'
sub_ids    = [
    d for d in os.listdir(preprc_dir)
    if d.startswith('sub-') and os.path.isdir(os.path.join(preprc_dir, d))
]
sub_ids.sort()

# glm directory
glm_dir = f'{main_dir}/analyses/lss_decision'
os.makedirs(glm_dir, exist_ok=True)

# One subject
run_lss_subject(
    sub_id=sub_ids[0],
    preprc_dir=preprc_dir,
    behav_dir=behav_dir,
    timing_file='timing.xlsx',
    glm_dir=glm_dir,
    tr=1.0,
    slice_time_ref=0.5,
    high_pass=1/128,
)

# All subjects
# for sub_id in sub_ids:
#     try:
#         run_lss_subject(
#             sub_id=sub_id,
#             preprc_dir=preprc_dir,
#             behav_dir=behav_dir,
#             timing_file=timing_file,
#             glm_dir=glm_dir,
#             tr=1.0,
#             slice_time_ref=0.5,
#             high_pass=1/128,
#         )
#     except Exception as e:
#         print(f"[{sub_id}] FAILED: {e}")
#         continue


[sub-18002]
  behav:     /Users/matty_gee/Desktop/Social/SocialCUD/data/preprocessed/behavior/sub-18002.xlsx
  func:      /Users/matty_gee/Desktop/Social/SocialCUD/data/preprocessed/fmriprep/sub-18002/func/sub-18002_task-socialnav_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii
  anat:      /Users/matty_gee/Desktop/Social/SocialCUD/data/preprocessed/fmriprep/sub-18002/anat/sub-18002_space-MNI152NLin2009cAsym_res-2_desc-preproc_T1w.nii.gz
  mask:      /Users/matty_gee/Desktop/Social/SocialCUD/data/preprocessed/fmriprep/sub-18002/func/sub-18002_task-socialnav_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii
  confounds: /Users/matty_gee/Desktop/Social/SocialCUD/data/preprocessed/fmriprep/sub-18002/func/sub-18002_task-socialnav_desc-confounds_timeseries.tsv
[sub-18002] fit   trial 001/63 (decision_num=1.0)


/Users/matty_gee/miniconda3/envs/social_cud/lib/python3.11/site-packages/nilearn/glm/first_level/design_matrix.py:416: UserWarning: The following conditions contain events with null duration:
- 'other_decision'

  matrix, names = _convolve_regressors(


[sub-18002] fit   trial 002/63 (decision_num=2.0)


/Users/matty_gee/miniconda3/envs/social_cud/lib/python3.11/site-packages/nilearn/glm/first_level/design_matrix.py:416: UserWarning: The following conditions contain events with null duration:
- 'other_decision'

  matrix, names = _convolve_regressors(


ImageFileError: Cannot work out file type of "/Users/matty_gee/Desktop/Social/SocialCUD/data/preprocessed/fmriprep/sub-18002/func/sub-18002_task-socialnav_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii"